In [0]:
%pip install /Workspace/Users/neil.braun@mirakl.com/.bundle/fast-gnn-benchmark/dev/files
dbutils.library.restartPython()

In [ ]:
from fast_gnn_benchmark.trainer import load_model_from_checkpoint
from fast_gnn_benchmark.data.dataset.coview_mdm import CoViewMDMDataset

import os
import torch
import boto3, json, io
import pyspark.sql.functions as F
from pyspark.sql import DataFrame, Row
from IPython.display import display, HTML

In [ ]:
from datetime import datetime


def list_checkpoints(ckpt_dir: str) -> list[tuple[float, str]]:
    """Liste les checkpoints de ckpt_dir, du plus recent au plus ancien, et les affiche.

    Args:
        ckpt_dir: Dossier de checkpoints (.ckpt) a lister.

    Returns:
        Liste de tuples (mtime, nom_fichier), triee par mtime decroissant.
    """
    files = [
        (os.path.getmtime(os.path.join(ckpt_dir, f)), f)
        for f in os.listdir(ckpt_dir)
        if f.endswith(".ckpt")
    ]

    for mtime, fname in sorted(files, reverse=True):
        print(f"{datetime.fromtimestamp(mtime):%Y-%m-%d %H:%M:%S}  {fname}")

    return files


ckpt_dir = "/dbfs/tmp/nbraun/checkpoints/coview-mdm-prototype"

files = list_checkpoints(ckpt_dir)

In [ ]:
from fast_gnn_benchmark.models.link_prediction import LinkPredictionModel


def load_best_checkpoint(
    ckpt_dir: str, files: list[tuple[float, str]], device: str
) -> tuple[LinkPredictionModel, str]:
    """Charge le checkpoint le plus recent parmi `files` (liste (mtime, nom) construite par la
    cellule precedente) et affiche les infos de diagnostic associees.

    save_top_k=1 sur val/mrr_trigger : le dossier ne contient que le checkpoint du meilleur
    epoch. On le resout dynamiquement, son nom dependant de l'epoch atteint -- pas de nom de
    fichier en dur ici.

    Args:
        ckpt_dir: Dossier contenant les checkpoints.
        files: Liste (mtime, nom_fichier) des checkpoints, construite par list_checkpoints.
        device: Device torch ("cuda" ou "cpu") sur lequel charger le modele.

    Returns:
        Tuple (model, checkpoint_path) : le modele charge (mode eval, sur device) et le
        chemin du checkpoint retenu.
    """
    assert files, f"aucun .ckpt dans {ckpt_dir}: lancer l'entrainement avec gae_gcn_coview_mdm_prototype.yml"
    checkpoint_path = os.path.join(ckpt_dir, sorted(files, reverse=True)[0][1])
    print(f"checkpoint: {checkpoint_path}")

    raw_ckpt = torch.load(checkpoint_path, map_location=device, weights_only=False)

    model = LinkPredictionModel.load_from_checkpoint(checkpoint_path, map_location=device, weights_only=False)
    model.eval()
    model.to(device)

    print("epoch:", raw_ckpt["epoch"])
    print("global_step:", raw_ckpt["global_step"])
    assert f"epoch={raw_ckpt['epoch']}-step={raw_ckpt['global_step']}" in checkpoint_path

    print(model.hparams.model_parameters)

    for cb_state in raw_ckpt["callbacks"].values():
        if "best_model_score" in cb_state:
            print("best_model_score (val/mrr_trigger):", cb_state["best_model_score"])

    print("nb params:", sum(p.numel() for p in model.parameters()))

    return model, checkpoint_path


device = "cuda" if torch.cuda.is_available() else "cpu"
model, checkpoint_path = load_best_checkpoint(ckpt_dir, files, device)

## Chargement du dataset et des mappings prototype

In [ ]:
def load_prototype_artifacts(bucket: str, prefix: str) -> tuple[CoViewMDMDataset, dict, dict, dict]:
    """Charge le dataset et les mappings prototype depuis S3.

    Artefacts prototype : exec2code y couvre toutes les executions du split, donc les exec_code
    diffèrent de ceux du pipeline principal. Ne jamais melanger avec data.pt / exec_mappings.json.

    Args:
        bucket: Bucket S3 contenant les artefacts prototype.
        prefix: Prefixe S3 commun a data_prototype.pt, node2idx_prototype.json et
            exec_mappings_prototype.json.

    Returns:
        Tuple (dataset, node2idx, idx2node, exec_mappings) : le dataset CoViewMDMDataset,
        les mappings internalId <-> node_id, et exec_mappings (code2exec par split
        train/val/test).
    """
    dataset = CoViewMDMDataset(bucket=bucket, s3_key=f"{prefix}/data_prototype.pt")

    s3 = boto3.client("s3")

    node2idx_raw = json.loads(
        s3.get_object(Bucket=bucket, Key=f"{prefix}/node2idx_prototype.json")["Body"].read()
    )
    node2idx = {int(k): v for k, v in node2idx_raw.items()}
    idx2node = {v: k for k, v in node2idx.items()}

    exec_mappings = json.loads(
        s3.get_object(Bucket=bucket, Key=f"{prefix}/exec_mappings_prototype.json")["Body"].read()
    )

    return dataset, node2idx, idx2node, exec_mappings


BUCKET = "mirakl-data-science-tmp2"
PREFIX = "nbraun/datasets/coview-mdm"

dataset, node2idx, idx2node, exec_mappings = load_prototype_artifacts(BUCKET, PREFIX)

print(f"num_nodes: {dataset.num_nodes}")
print(f"node2idx entries: {len(node2idx)}")
for split in ["train", "val", "test"]:
    print(f"{split}: {len(exec_mappings[split]['code2exec'])} executions")

## Tables prototype

In [0]:
prod_results_val_prototype = spark.read.parquet(
    f"s3://{BUCKET}/{PREFIX}/prod_results_val_prototype.parquet"
)
sessions_raw_val_prototype = spark.read.parquet(
    f"s3://{BUCKET}/{PREFIX}/sessions_raw_val_prototype.parquet"
)

print(f"prod_results_val_prototype: {prod_results_val_prototype.count()} triggers")
prod_results_val_prototype.printSchema()
display(prod_results_val_prototype.limit(1))

print(f"sessions_raw_val_prototype: {sessions_raw_val_prototype.count()} triggers")
display(sessions_raw_val_prototype.limit(1))

## Embeddings de nœuds, calculés une seule fois et réutilisés par tous les triggers

In [ ]:
import torch
from torch_geometric.data import Data
from torch_geometric.transforms import ToSparseTensor


def compute_node_embeddings(
    dataset: CoViewMDMDataset, model: LinkPredictionModel, device: str
) -> torch.Tensor:
    """Forward pass GNN complet (embedder + backbone), calcule une seule fois et reutilise par
    tous les triggers.

    Args:
        dataset: Dataset dont dataset.data.x et dataset.data.edge_index alimentent le forward.
        model: Modele charge (LinkPredictionModel) ; son embedder et son backbone sont utilises.
        device: Device torch sur lequel calculer les embeddings.

    Returns:
        Tensor des embeddings de tous les noeuds du graphe (un par node_id de node2idx),
        sortie du backbone -- pas encore projetes dans l'espace du classifieur.
    """
    N = dataset.num_nodes
    adj_t = ToSparseTensor()(Data(edge_index=dataset.data.edge_index, num_nodes=N)).adj_t.to(device)
    assert adj_t.layout == torch.sparse_csr, f"layout inattendu: {adj_t.layout}"

    for m in model.model.backbone.modules():
        if hasattr(m, "_cached_edge_index"):
            m._cached_edge_index = None
        if hasattr(m, "_cached_adj_t"):
            m._cached_adj_t = None

    torch.cuda.empty_cache()

    model.eval()

    with torch.no_grad():
        x = model.model.embedder(dataset.data.x.to(device))
        x = model.model.backbone(x, adj_t)

    return x


batch_size = 8192

x = compute_node_embeddings(dataset, model, device)

print(f"x shape: {tuple(x.shape)}")

## Extraction des triggers — un seul chemin

In [ ]:
def extract_all_triggers(prod_results: DataFrame) -> list[dict]:
    """Tous les triggers de la table prod prototype.

    Remplace extract_triggers (qui partait de val_split["edge"], donc des seules executions ayant
    au moins un positif dans le top-12 prod) et extract_negative_only_triggers. Le tenseur d'edges
    n'est plus utilise ici: la population d'analyse est definie independamment de l'etiquetage.

    Args:
        prod_results: DataFrame prod prototype, doit contenir exec_code et trigger_internal_id.

    Returns:
        Liste de dicts (un par exec_code distinct), avec les cles exec_code,
        trigger_internal_id et trigger_node_id. Un trigger dont le produit declencheur
        n'a pas de node_id dans node2idx (hors graphe) est absent du resultat.
    """
    rows = (
        prod_results
        .select("exec_code", "trigger_internal_id")
        .dropDuplicates(["exec_code"])
        .collect()
    )

    triggers = []
    skipped_no_node = 0

    for row in rows:
        trigger_internal_id = int(row["trigger_internal_id"])

        if trigger_internal_id not in node2idx:
            skipped_no_node += 1
            continue

        triggers.append({
            "exec_code": row["exec_code"],
            "trigger_internal_id": trigger_internal_id,
            "trigger_node_id": node2idx[trigger_internal_id],
        })

    print(f"triggers hors graphe (produit sans node_id): {skipped_no_node}/{len(rows)}")

    return triggers


triggers = extract_all_triggers(prod_results_val_prototype)

print(f"{len(triggers)} triggers a scorer")

In [ ]:
def extract_categories(triggers: list[dict]) -> list[dict]:
    """Attache a chaque trigger sa t2s_best_fitting_category. trigger_internal_id est deja
    renseigne par extract_all_triggers, contrairement a la version non-prototype."""

    unique_internal_ids = {t["trigger_internal_id"] for t in triggers}

    df_trigger_ids = spark.createDataFrame(
        [(str(i),) for i in unique_internal_ids], schema="internalId string"
    )

    df_categories = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(F.col("customer_short_name") == "maisons-du-monde")
        .join(F.broadcast(df_trigger_ids), on="internalId", how="left_semi")
        .select("internalId", F.col("t2s_best_fitting_category")[0].alias("category"))
        .dropDuplicates(["internalId"])
        .collect()
    )

    category_by_internal_id = {
        int(row["internalId"]): row["category"]
        for row in df_categories
        if row["category"] is not None
    }

    for trigger in triggers:
        trigger["category"] = category_by_internal_id.get(trigger["trigger_internal_id"])

    return triggers


triggers = extract_categories(triggers)

missing = sum(1 for t in triggers if t["category"] is None)
print(f"triggers sans categorie: {missing}/{len(triggers)}")

In [ ]:
def build_candidates_by_category(triggers: list[dict]) -> dict:

    unique_categories = {t["category"] for t in triggers if t["category"] is not None}

    df_category_products = (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(
            F.col("t2s_best_fitting_category")[0].isin(list(unique_categories))
            & (F.col("customer_short_name") == "maisons-du-monde")
        )
        .select("internalId", F.col("t2s_best_fitting_category")[0].alias("category"))
        .dropDuplicates(["internalId"])
        .collect()
    )

    internal_ids_by_category = {c: [] for c in unique_categories}
    for row in df_category_products:
        internal_ids_by_category[row["category"]].append(int(row["internalId"]))

    candidate_node_ids_by_category = {}
    for category, internal_ids in internal_ids_by_category.items():
        node_ids = torch.tensor([node2idx[i] for i in internal_ids if i in node2idx])
        candidate_node_ids_by_category[category] = torch.unique(node_ids)  # plusieurs internalId peuvent partager le même node_id

    return candidate_node_ids_by_category

candidates_by_category = build_candidates_by_category(triggers)

pool_sizes = [c.numel() for c in candidates_by_category.values()]
print(f"nombre de catégories: {len(candidates_by_category)}")
print(f"taille moyenne du pool de candidats par catégorie: {sum(pool_sizes) / len(pool_sizes):.0f}")

## Inférence sur tous les triggers

In [ ]:
TOP_K = 12


def run_inference(triggers: list[dict], candidates_by_category: dict) -> list[dict]:

    skipped_no_candidates = 0

    with torch.no_grad():
        for trigger in triggers:
            if trigger["category"] is None:
                skipped_no_candidates += 1
                continue

            candidates_to_score = candidates_by_category[trigger["category"]]
            candidates_to_score = candidates_to_score[candidates_to_score != trigger["trigger_node_id"]]

            if candidates_to_score.numel() == 0:
                skipped_no_candidates += 1
                continue

            target_edges = torch.stack([
                torch.full_like(candidates_to_score, trigger["trigger_node_id"]),
                candidates_to_score,
            ])

            logits_chunks = []
            for start in range(0, target_edges.shape[1], batch_size):
                chunk = target_edges[:, start:start + batch_size].to(device)
                logits_chunks.append(model.model.classifier(x, x, chunk))

            logits = torch.cat(logits_chunks)
            k = min(TOP_K, logits.numel())  
            top_logits, top_positions = torch.topk(logits, k)
            top_node_ids = candidates_to_score[top_positions.cpu()]

            node_ids_list = top_node_ids.tolist()
            scores_list = top_logits.cpu().tolist()

            top12_model = [
                {
                    "internalId": int(idx2node[node_id]),
                    "score": float(score),
                    "rank": rank,
                }
                for rank, (node_id, score) in enumerate(zip(node_ids_list, scores_list), start=1)
            ]

            trigger["candidate_pool_size"] = candidates_to_score.numel()
            trigger["top12_model"] = top12_model

    print(f"triggers non scores (categorie manquante ou pool vide): {skipped_no_candidates}/{len(triggers)}")

    return triggers

In [ ]:
triggers = run_inference(triggers, candidates_by_category)

n_scored = sum(1 for t in triggers if t.get("top12_model"))
print(f"{n_scored}/{len(triggers)} triggers scores")

## Résultats du modèle

In [ ]:
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, DoubleType

MODEL_ROWS_PATH = f"s3://{BUCKET}/{PREFIX}/_staging_model_rows_val_prototype.parquet"
BATCH_TRIGGERS = 20_000

model_rows_schema = StructType([
    StructField("exec_code", LongType(), True),
    StructField("internal_id", LongType(), True),
    StructField("score", DoubleType(), True),
    StructField("rank", IntegerType(), True),
])

keys_schema = StructType([
    StructField("exec_code", LongType(), True),
    StructField("trigger_internal_id", LongType(), True),
])

EMPTY_PRODUCTS = F.array().cast("array<struct<internal_id:bigint,score:double,rank:int>>")


def stage_model_rows(triggers: list[dict], path: str, batch_triggers: int = BATCH_TRIGGERS) -> None:
    """Ecrit les lignes plates par lots. Seul le premier lot ecrase, les suivants ajoutent."""
    mode = "overwrite"

    for start in range(0, len(triggers), batch_triggers):
        rows = [
            (t["exec_code"], v["internalId"], v["score"], v["rank"])
            for t in triggers[start:start + batch_triggers]
            for v in t.get("top12_model", [])
        ]
        if not rows:
            continue

        (
            spark.createDataFrame(rows, schema=model_rows_schema, verifySchema=False)
            .write.mode(mode).parquet(path)
        )
        print(f"  lot {start}-{min(start + batch_triggers, len(triggers))}: {len(rows)} lignes")
        mode = "append"

    if mode == "overwrite":  
        spark.createDataFrame([], schema=model_rows_schema).write.mode("overwrite").parquet(path)
        print("  aucun trigger score, parquet vide ecrit")


def build_model_results(triggers: list[dict], sessions_raw: DataFrame) -> DataFrame:

    stage_model_rows(triggers, MODEL_ROWS_PATH)

    df_keys = spark.createDataFrame(
        [(t["exec_code"], t["trigger_internal_id"]) for t in triggers],
        schema=keys_schema,
        verifySchema=False,
    )

    df_products = (
        spark.read.parquet(MODEL_ROWS_PATH)
        .groupBy("exec_code")
        .agg(F.sort_array(F.collect_list(F.struct("rank", "internal_id", "score"))).alias("ranked"))
        .withColumn(
            "products_returned",
            F.transform(
                "ranked",
                lambda r: F.struct(
                    r["internal_id"].alias("internal_id"),
                    r["score"].alias("score"),
                    r["rank"].alias("rank"),
                ),
            ),
        )
        .select("exec_code", "products_returned")
    )

    session_ids_by_exec_code = (
        sessions_raw
        .select("exec_code", F.col("session_products.internal_id").alias("session_ids"))
        .dropDuplicates(["exec_code"])
    )

    return (
        df_keys
        .join(df_products, on="exec_code", how="left")
        .join(session_ids_by_exec_code, on="exec_code", how="left")
        .withColumn("products_returned", F.coalesce(F.col("products_returned"), EMPTY_PRODUCTS))
        .withColumn("session_ids", F.coalesce(F.col("session_ids"), F.array().cast("array<bigint>")))
        .withColumn(
            "positives",
            F.filter("products_returned", lambda p: F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
        .withColumn(
            "negatives",
            F.filter("products_returned", lambda p: ~F.array_contains(F.col("session_ids"), p["internal_id"])),
        )
        .select("exec_code", "trigger_internal_id", "positives", "negatives", "products_returned")
    )


model_results_val_prototype = build_model_results(triggers, sessions_raw_val_prototype).cache()

print(f"model_results_val_prototype: {model_results_val_prototype.count()} triggers")

model_results_val_prototype.printSchema()

display(model_results_val_prototype.limit(1))

In [ ]:
model_results_val_prototype.write.mode("overwrite").parquet(
    f"s3://{BUCKET}/{PREFIX}/model_results_val_prototype.parquet"
)

print("model_results_val_prototype.parquet uploaded")

In [ ]:
import gc

sample_trigger = next(t for t in triggers if t.get("top12_model"))

del triggers, candidates_by_category
gc.collect()

print("triggers et candidates_by_category liberes")
print(f"trigger conserve pour le controle visuel: exec_code={sample_trigger['exec_code']}")

## Contrôle visuel sur un trigger

In [0]:
def get_customer_db_name(customer_shortname: str) -> str:
    df_customer = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_gold_customer")
        .where(F.col("shortName") == customer_shortname)
        .select(F.col("databaseName").alias("db_name"))
    )
    return df_customer.collect()[0]["db_name"]


def display_product_model(internal_ids: list[int], db_name: str, image_width: int = 150) -> None:
    if not internal_ids:
        print("(aucun produit)")
        return

    ranks_df = spark.createDataFrame(
        [Row(internalId=i, topk_rank=rank) for rank, i in enumerate(internal_ids, start=1)]
    )

    df_products = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_mongo_product_0_current")
        .filter(F.col("db_name") == db_name)
        .select(F.col("internalId").cast("bigint").alias("internalId"), "name", "imageUrl")
        .dropDuplicates(["internalId"])
    )

    df_products_info = (
        ranks_df
        .join(df_products, on="internalId", how="left")
        .select("topk_rank", "internalId", "name", "imageUrl")
        .orderBy("topk_rank")
    )

    for row in df_products_info.collect():
        if row["imageUrl"] is None:
            print(f"{row['topk_rank']}. internalId={row['internalId']} — pas d'image trouvee")
            continue
        display(HTML(f"<p><b>{row['topk_rank']}.</b> {row['name']} (internalId={row['internalId']})</p>"))
        display(HTML(f'<img src="{row["imageUrl"]}" width="{image_width}">'))


db_name = get_customer_db_name("maisons-du-monde")

In [ ]:
def ids_in_rank_order(products) -> list[int]:
    return [int(p["internal_id"]) for p in sorted(products, key=lambda p: p["rank"])]


def build_comparison(sample_trigger: dict, prod_results: DataFrame) -> tuple[list[int], list[int], list[int]]:
    sample_exec_code = sample_trigger["exec_code"]

    sample_prod = (
        prod_results
        .filter(F.col("exec_code") == sample_exec_code)
        .collect()[0]
    )

    prod_display_ids = ids_in_rank_order(sample_prod["products_returned_display"])
    prod_relevance_ids = ids_in_rank_order(sample_prod["products_returned_relevance"])
    model_ids = [int(p["internalId"]) for p in sample_trigger["top12_model"]]

    print(f"exec_code {sample_exec_code} -> executionId prod: {sample_prod['execution_id']}")
    print(f"produit declencheur: internalId={sample_trigger['trigger_internal_id']}")
    print(f"categorie: {sample_trigger['category']} | pool de candidats: {sample_trigger['candidate_pool_size']}")
    print()
    print(f"prod display  : {prod_display_ids}")
    print(f"prod relevance: {prod_relevance_ids}")
    print(f"modele        : {model_ids}")
    print()
    print(f"overlap display/relevance : {len(set(prod_display_ids) & set(prod_relevance_ids))}")
    print(f"overlap modele/display    : {len(set(model_ids) & set(prod_display_ids))}")
    print(f"overlap modele/relevance  : {len(set(model_ids) & set(prod_relevance_ids))}")

    return prod_display_ids, prod_relevance_ids, model_ids


prod_display_ids, prod_relevance_ids, model_ids = build_comparison(sample_trigger, prod_results_val_prototype)

In [0]:
for title, ids in [
    ("Prod — 12 premiers dans l'ordre d'affichage", prod_display_ids),
    ("Prod — 12 premiers par relevanceScore", prod_relevance_ids),
    ("Modele — top 12", model_ids),
]:
    display(HTML(f"<h3>{title}</h3>"))
    display_product_model(ids, db_name=db_name)